In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import sklearn

In [2]:
df=pd.read_csv('/home/yazid/sklearn/medical_cost/data/medical_insurance.csv')

Let's start with some data exploration:

In [3]:
len(df)

2772

In [4]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [5]:
df.isna().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Let's fix dtypes:

In [6]:
df['smoker']=df['smoker'].map({'yes':1,'no':0})
df['sex']=df['sex'].map({'female':1,'male':0})

Let's create new features:

Let's create some columns (Feature engineering):

In [7]:
df['high bmi']=(df['bmi']>30).astype(bool)
df['senior']=(df['age']>=65).astype(bool)
df['young']=(df['age']<=30).astype(bool)
df['healthy bmi']=(df['bmi']<25).astype(bool)


In [8]:
df.columns.tolist()

['age',
 'sex',
 'bmi',
 'children',
 'smoker',
 'region',
 'charges',
 'high bmi',
 'senior',
 'young',
 'healthy bmi']

Let's create a baseline model:

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
preprocessor=ColumnTransformer(transformers=[('reg_encoder',OneHotEncoder(drop='first'),['region'])],remainder='passthrough')
X=df.drop('charges',axis=1)
y=df['charges']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train_proc=preprocessor.fit_transform(X_train)
X_test_proc=preprocessor.transform(X_test)

In [10]:
from sklearn.ensemble import RandomForestRegressor
v0=RandomForestRegressor()
v0.fit(X_train_proc,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the numb

In [11]:
from sklearn.metrics import mean_absolute_error
y_pred_0=v0.predict(X_test_proc)
mae_0=mean_absolute_error(y_pred_0,y_test)
(mae_0/y_test.median())*100 #We get the pourcentage of our mae :

np.float64(14.413990983814697)

Our Mae is high let's do a quick randomized search to improve our model's performance:

In [12]:
from sklearn.model_selection import RandomizedSearchCV
param_grid={'n_estimators':[300,500,700],
            'max_depth':[10,20,30,40,50,None],
           'min_samples_split':[2,5,10,15],
            'min_samples_leaf':[1,2,4,6,10],
'max_features':['sqrt','log2',0.5,0.8,None]}
randomsearch=RandomizedSearchCV(estimator=v0,param_distributions=param_grid,n_iter=80,cv=5,scoring='neg_mean_absolute_error',random_state=42,n_jobs=-1)
randomsearch.fit(X_train_proc,y_train)
print(f'Best params:{randomsearch.best_params_}')
#Best params:{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 30}

Best params:{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 30}


In [13]:
def feature_importance(model):
    feature_names=preprocessor.get_feature_names_out()
    importances=pd.DataFrame({'Feature':feature_names,
                             'Importance':model.feature_importances_}).sort_values('Importance',ascending=False).reset_index(drop=True)
    return importances

In [14]:
v1=RandomForestRegressor(n_estimators=500,max_depth=30,min_samples_split=2,min_samples_leaf=1,max_features=0.5,random_state=42,n_jobs=-1)
v1.fit(X_train_proc,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",30
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max

In [15]:
y_pred_1=v1.predict(X_test_proc)
mae_1=mean_absolute_error(y_test,y_pred_1)
(mae_1/y_test.median())*100

np.float64(14.30921492042587)

In [16]:
feature_importance(v1)

,Feature,Importance
0,remainder__smoker,0.647889
1,remainder__bmi,0.119161
2,remainder__age,0.109819
3,remainder__high bmi,0.052841
4,remainder__children,0.019250
5,remainder__young,0.017604
6,remainder__healthy bmi,0.008323
7,remainder__sex,0.007158
8,reg_encoder__region_northwest,0.006338
9,reg_encoder__region_southeast,0.006313


In [17]:
df.drop('senior',axis=1,inplace=True) #Since senior column scored terribly:

In [18]:
df['bmi*smoker']=(df['bmi']*df['smoker'])

In [19]:
X=df.drop('charges',axis=1)
y=df['charges']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
preprocessor=ColumnTransformer([('one_hot',OneHotEncoder(drop='first'),['region'])],remainder='passthrough')
X_train_proc=preprocessor.fit_transform(X_train)
X_test_proc=preprocessor.transform(X_test)

In [21]:
v3=RandomForestRegressor(n_estimators=500,min_samples_leaf=1,min_samples_split=2,max_features=0.5,max_depth=30,random_state=42,n_jobs=-1)
v3.fit(X_train_proc,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",30
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max

In [22]:
y_pred_3=v3.predict(X_test_proc)
mae_3=mean_absolute_error(y_pred_3,y_test)
(mae_3/y_test.median())*100

np.float64(14.130142247069452)

In [23]:
feature_importance(v3)

,Feature,Importance
0,remainder__bmi*smoker,0.476202
1,remainder__smoker,0.261957
2,remainder__age,0.104525
3,remainder__bmi,0.076032
4,remainder__high bmi,0.019127
5,remainder__young,0.018836
6,remainder__children,0.017365
7,remainder__sex,0.006065
8,one_hot__region_northwest,0.006021
9,one_hot__region_southeast,0.005184


Price columns is right skewed let's fix this:

In [24]:
df.columns.tolist()
df['log charges']=np.log1p(df['charges'])

In [25]:
df.columns.tolist()

['age',
 'sex',
 'bmi',
 'children',
 'smoker',
 'region',
 'charges',
 'high bmi',
 'young',
 'healthy bmi',
 'bmi*smoker',
 'log charges']

In [26]:
preprocessor=ColumnTransformer([('one_hot',OneHotEncoder(drop='first'),['region'])],remainder='passthrough')
X=df.drop(['log charges','charges'],axis=1)
y=df['log charges']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train_proc=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

In [27]:
v4=RandomForestRegressor(n_estimators=500,max_depth=30,min_samples_split=2,min_samples_leaf=1,max_features=0.5,random_state=42,n_jobs=-1)
v4.fit(X_train_proc,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",30
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max

In [28]:
y_pred_4=v4.predict(X_test_proc)
mae_4=mean_absolute_error(np.expm1(y_pred_4),np.expm1(y_test))
mae_4

1224.9688505791255

To wrap things up let's try xgbclassifier as our final model for this project:

In [34]:
from xgboost import XGBRegressor
param_distrb_xgb={'n_estimators':[300,500,700,1000],
                 'learning_rate':[0.01,0.05,0.1,0.2],
                 'max_depth':[3,4,5,6,7,8],
                 'min_child_weight':[1,3,5,7],
                 'subsample':[0.6,0.8,1.0],
                 'colsample_bytree':[0.6,0.8,1.0],
                 'reg_alpha':[0,0.1,1,10],
                 'reg_lambda':[0.1,1,5,10],
                 'gamma':[0,0.1,0.5]}
xgb_base=XGBRegressor(random_state=42,n_jobs=-1,tree_method='hist')
random_search_xgb=RandomizedSearchCV(estimator=xgb_base,param_distributions=param_distrb_xgb,n_iter=50,cv=5,scoring='neg_mean_absolute_error',
                                    random_state=42,n_jobs=-1,verbose=1)
random_search_xgb.fit(X_train_proc,y_train)
print(f"Best xgb-params:{random_search_xgb.best_params_}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best xgb-params:{'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 1.0}


In [37]:
final=random_search_xgb.best_estimator_
y_pred_final=final.predict(X_test_proc)
mae_final=mean_absolute_error(np.expm1(y_pred_final),np.expm1(y_test))
mae_final

1026.1702732707633

Let's look at the % of our latest mae:

In [39]:
(mae_final/(df['charges'].median()))*100

np.float64(10.99505727504601)